<a href="https://colab.research.google.com/github/smonodeep/DynamicValuationGovernance/blob/main/0.Data_Prep_CityWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data prep code

In [40]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import pandas_datareader.data as web
import datetime
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

In [41]:
rbi_hpi_path = '/content/House Price Index Publication_RBI.xlsx'

try:
    # Read only the specified range for the 2010-11 HPI block
    df_hpi_raw = pd.read_excel(rbi_hpi_path, sheet_name='House Price Index', header=None, skiprows=27, nrows=61, usecols='B:M')
    print(f"Successfully loaded RBI HPI Excel from {rbi_hpi_path}")
    # Display first few rows to verify loading
    print("Raw HPI data from specified range:")
    display(df_hpi_raw.head())
except FileNotFoundError:
    print(f"Error: RBI HPI Excel file not found at {rbi_hpi_path}")
    df_hpi_raw = pd.DataFrame()
except Exception as e:
    print(f"Error loading RBI HPI Excel: {e}")
    df_hpi_raw = pd.DataFrame()

Successfully loaded RBI HPI Excel from /content/House Price Index Publication_RBI.xlsx
Raw HPI data from specified range:


,1,2,3,4,5,6,7,8,9,10,11,12
0,NaN,Ahmedabad,Bangalore,Chennai*,Delhi,Jaipur,Kanpur,Kochi,Kolkata,Lucknow,Mumbai,All India
1,Q1.2010-11,93.218679,98.639697,102.731254,100.721966,95.312154,91.735054,89.611743,77.879138,88.82087,90.626706,94.239884
2,Q2.2010-11,102.54263,97.902562,109.501643,95.560697,99.047963,99.446163,92.376464,103.191573,98.671878,99.717032,99.811033
3,Q3.2010-11,101.99031,97.935658,94.647711,92.105168,103.574958,103.686822,113.849679,106.617769,104.745752,100.879902,99.404514
4,Q4.2010-11,102.248381,105.522082,93.119392,112.058101,102.064925,105.13196,104.162114,112.31152,107.761499,108.77636,106.63229


In [42]:
if not df_hpi_raw.empty:
    df_hpi = df_hpi_raw.copy()

    # The first row contains city names, the first column contains quarter labels
    city_names = df_hpi.iloc[0, 1:].tolist()
    # quarter_labels are in the first column, from the second row onwards
    # No need to extract quarter_labels explicitly here as they will be in the 'quarter' column after column setting

    # Set column names
    df_hpi.columns = ['quarter'] + city_names

    # Drop the first row (city names are now headers)
    df_hpi = df_hpi.drop(df_hpi.index[0])

    # Reset index
    df_hpi = df_hpi.reset_index(drop=True)

    # Convert wide format to long format
    df_hpi_long = df_hpi.melt(id_vars=['quarter'], var_name='city', value_name='hpi')

    # Ensure quarter and city are strings and hpi is numeric
    df_hpi_long['quarter'] = df_hpi_long['quarter'].astype(str)
    df_hpi_long['city'] = df_hpi_long['city'].astype(str)
    df_hpi_long['hpi'] = pd.to_numeric(df_hpi_long['hpi'], errors='coerce')

    print("HPI data after initial cleaning and melt:")
    display(df_hpi_long.head())
else:
    print("df_hpi_raw is empty, skipping HPI cleaning.")
    df_hpi_long = pd.DataFrame()

HPI data after initial cleaning and melt:


,quarter,city,hpi
0,Q1.2010-11,Ahmedabad,93.218679
1,Q2.2010-11,Ahmedabad,102.542630
2,Q3.2010-11,Ahmedabad,101.990310
3,Q4.2010-11,Ahmedabad,102.248381
4,Q1.2011-12,Ahmedabad,121.296551


In [43]:
if not df_hpi_long.empty:
    # 1. Lowercase city names, remove '*' and strip whitespace
    df_hpi_long['city'] = df_hpi_long['city'].str.lower().str.replace('*', '', regex=False).str.strip()

    # 2. Remove 'all india' and NaN HPI values
    df_hpi_long = df_hpi_long[df_hpi_long['city'] != 'all india']
    df_hpi_long = df_hpi_long.dropna(subset=['hpi'])

    # 3. Retain only specified cities
    common_cities = ['bangalore', 'chennai', 'delhi', 'mumbai']
    df_hpi_long = df_hpi_long[df_hpi_long['city'].isin(common_cities)].reset_index(drop=True)

    print("HPI data after standardization and city filtering:")
    print(f"Shape: {df_hpi_long.shape}")
    print(f"Unique cities: {df_hpi_long['city'].unique()}")
    display(df_hpi_long.head())
else:
    print("df_hpi_long is empty, skipping HPI standardization.")

HPI data after standardization and city filtering:
Shape: (240, 3)
Unique cities: ['bangalore' 'chennai' 'delhi' 'mumbai']


,quarter,city,hpi
0,Q1.2010-11,bangalore,98.639697
1,Q2.2010-11,bangalore,97.902562
2,Q3.2010-11,bangalore,97.935658
3,Q4.2010-11,bangalore,105.522082
4,Q1.2011-12,bangalore,110.679615
